## Case Table

**Modification Objective:** Create a case-level representation of the event log in which each case is represented by a single row containing its unique identifier and case-level attributes.

**Motivation:** Case-centric event logs typically represent cases through multiple event rows, which can make case-level properties inconvenient to inspect or analyze directly. Creating a case table provides a compact representation of one row per case and makes case-level attributes readily available for subsequent exploration, comparison, or analysis.

**Precondition:** A list of attribute columns to be included in the case table. For each selected attribute, either a unique value per case must exist or a rule for selecting the value to retain must be specified.

**Approach:** Aggregate the event log at the case level by identifying attributes that represent case-level information and consolidate their values into a single record for each case.

**Output:** A case table containing one row per case and columns for the identified case-level attributes.

**Implemented Example**: RTFM log - Almost all attribute columns are local attributes specific to a unique activity and case-level attribute candidates and can therefore be stored in a case table (cf., patterns *attribute_scope* and *log_and_case_attribute_candidates*). Additionally, the attribute "paymentAmount" can be considered as the full payment amount per case by adding all paymentAmounts per case.

In [ ]:
import pandas as pd
import pm4py

from itables import show

# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

# Attributes to include in the case table
CASE_TABLE_COLUMNS = [
    "article",
    "expense",
    "lastSent",
    "matricola",
    "notificationType",
    "points",
    "vehicleClass",
    "paymentAmount"
]

AGGREGATION_RULES = {"paymentAmount": "sum"}

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

In [ ]:
required_columns = [CASE_ID] + CASE_TABLE_COLUMNS

#validate that all columns exist and satisfy the precondition
missing_columns = [
    column
    for column in required_columns
    if column not in event_log.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required column(s): {missing_columns}"
    )

attribute_validation = []

for attribute in CASE_TABLE_COLUMNS:
    distinct_values_per_case = (
        event_log
        .groupby(CASE_ID)[attribute]
        .nunique(dropna=True)
    )

    max_values_per_case = distinct_values_per_case.max()
    cases_with_multiple_values = (distinct_values_per_case > 1).sum()

    attribute_validation.append({
        "Attribute": attribute,
        "Max Values per Case": max_values_per_case,
        "Cases with Multiple Values": cases_with_multiple_values,
        "Rule Specified": attribute in AGGREGATION_RULES
    })

attribute_validation = pd.DataFrame(attribute_validation)

display(attribute_validation)

In [ ]:
def first_non_null(series):
    """Return the first non-missing value, or NA if no value is available."""
    
    values = series.dropna()
    
    if values.empty:
        return pd.NA
    
    return values.iloc[0]

In [ ]:
aggregation_rules = {
    attribute: AGGREGATION_RULES.get(attribute, first_non_null)
    for attribute in CASE_TABLE_COLUMNS
}

In [ ]:
case_table = (
    event_log
    .groupby(CASE_ID, as_index=False)
    .agg(aggregation_rules)
    #.rename(columns={"paymentAmount": "TotalAmountPaid" #optional
)

display(case_table.head())

In [ ]:
#might take a bit to load

show(
    case_table,
    paging=True,
    searching=True,
    ordering=True,
    scrollX=True,
    pageLength=30,
    maxBytes=0
)